# Oto-1 training on Colab (plan 0004, Phase 1)

Runs `tools/train` on a Colab GPU (A100 / L4 / T4). Shards are staged from **Cloudflare R2** or a **private Hugging Face dataset repo** to local NVMe (`/content/data`); never train from a Google Drive FUSE mount. Checkpoints go to `/content/ckpt` every 20 min and are mirrored to R2/HF, so a disconnected or 24 h-capped runtime **resumes automatically** when you re-run all cells.

1. Runtime → Change runtime type → GPU (A100 if available).
2. Add Colab secrets (🔑 sidebar): `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` (or `HF_TOKEN`), and `GH_TOKEN` if the repo is private.
3. Set the run config below, then Runtime → Run all.

In [ ]:
# --- run config -------------------------------------------------------------
ARCH = "a"                       # "a" = DFN-style hybrid, "b" = FastEnhancer-style
RUN = "clean-a-001"
DATA = "r2://oto-train/shards/v1/"   # or "hf://<user>/<repo>/shards/v1/"
RIRS = "r2://oto-train/rirs/v1/"
SYNC = f"r2://oto-train/runs/{RUN}/"
SMOKE = True                      # True: synthetic dry run, no datasets needed
CKPT_DIR = f"/content/ckpt/{'smoke-' if SMOKE else ''}{RUN}"
EXTRA = ""                        # e.g. f"--long-seq --init-from {CKPT_DIR}/best.pt" or "--isolate"
REPO = "https://github.com/<owner>/oto.git"

In [ ]:
import os, subprocess
from google.colab import userdata
for k in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "HF_TOKEN", "GH_TOKEN"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
url = REPO.replace("https://", f"https://{os.environ['GH_TOKEN']}@") if os.environ.get("GH_TOKEN") else REPO
if not os.path.exists("/content/oto"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/oto"], check=True)
%cd /content/oto/tools/train
# Colab already ships torch with CUDA; install the rest into the system env.
!pip install -q -e ".[upload]" pytest

In [ ]:
# Quick sanity: parity + export tests (CPU, ~1 min)
!python -m pytest -q tests/test_models.py tests/test_export.py
!python -m oto_train.models

In [ ]:
cmd = f"python -m oto_train.train --arch {ARCH} --ckpt-dir {CKPT_DIR} --ckpt-minutes 20"
if SMOKE:
    cmd += " --smoke"
else:
    cmd += (f" --data {DATA} --stage-dir /content/data --rir-data {RIRS} --sync {SYNC}"
            f" --batch-size 32 --seconds 4 --steps 300000 --workers 10 --compile")
cmd += " " + EXTRA
print(cmd)
!{cmd}

In [ ]:
# Best model exports (written on every validation improvement and mirrored to SYNC)
!ls -la {CKPT_DIR}
from oto_train.export_otow import read_otow
o = read_otow(f"{CKPT_DIR}/best.otow")
print(o.arch_id, len(o.tensors), o.sha256.hex(), o.config)